In [27]:
# Core RAG Frameworks
!pip install langchain-classic
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain
!pip install faiss-cpu
!pip install tiktoken
!pip install openai
!pip install grandalf
!pip install chromadb

In [28]:
# Import all depedencies for tools

import json
import os
from typing import List, Dict, Any
from langchain_core.tools import Tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Import the Python SDK for Colab secrets
from google.colab import userdata

In [29]:
import os

# Fetch the API key from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_APIKEY')

# Set up Environmental Variables
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

In [30]:
class SupportTicketTools:
    """Collection of tools for the support assistant agent."""
    def __init__(self, tickets_path: str = './sample_data/synthetic_tickets.json'):
        """
        Initialize ticket data + retrieval infrastructure once at startup.
        This keeps tool calls fast because embeddings/vector index are prepared
        ahead of time rather than rebuilt per request.
        """
        # Load tickets
        with open(tickets_path, 'r', encoding='utf-8') as f:
            self.tickets = json.load(f)
        # Create vector store for RAG
        self.embeddings = OpenAIEmbeddings(
            model=OPENAI_EMBEDDING_MODEL,
            openai_api_key=OPENAI_API_KEY
        )
        self.setup_vectorstore()

    def setup_vectorstore(self):
        """
        Build a semantic-search index from ticket text.
        Each ticket becomes one Document with metadata so agent responses can
        include both narrative context and structured filters/citations.
        """
        documents = []
        for ticket in self.tickets:
            content = f"""Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
Category: {ticket['category']}
Priority: {ticket['priority']}"""
            doc = Document(
                page_content=content,
                metadata={
                    'ticket_id': ticket['ticket_id'],
                    'category': ticket['category'],
                    'priority': ticket['priority'],
                    'title': ticket['title']
                }
            )
            documents.append(doc)
        # Persist locally so repeated runs reuse indexed data on disk.
        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            persist_directory="./agent_vectorstore"
        )

    def search_similar_tickets(self, query: str) -> str:
        """
        Search for similar support tickets using semantic similarity.
        Args:
            query: User's problem description or question
        Returns:
            Formatted string with relevant tickets
        """
        # Top-k semantic retrieval by embedding similarity.
        results = self.vectorstore.similarity_search(query, k=3)
        if not results:
            return "No similar tickets found."
        output = "Found similar tickets:\n\n"
        for i, doc in enumerate(results, 1):
            output += f"--- Ticket {i} ---\n"
            output += doc.page_content + "\n\n"
        return output

    def get_ticket_by_id(self, ticket_id: str) -> str:
        """
        Retrieve a specific ticket by its ID.
        Args:
            ticket_id: The ticket ID (e.g., TICK-001)
        Returns:
            Formatted ticket details or error message
        """
        # Normalize input to avoid case/whitespace mismatches.
        ticket_id = ticket_id.upper().strip()
        for ticket in self.tickets:
            if ticket['ticket_id'] == ticket_id:
                return f"""Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Created: {ticket['created_date']}
Resolved: {ticket['resolved_date']}"""
        return f"Ticket {ticket_id} not found in the database."

    def search_by_category(self, category: str) -> str:
        """
        Find all tickets in a specific category.
        Args:
            category: Category name (e.g., Authentication, Database, Payment)
        Returns:
            List of tickets in the category
        """
        # Normalize category string for case-insensitive exact matching.
        category = category.strip()
        matching = [t for t in self.tickets if t['category'].lower() == category.lower()]
        if not matching:
            available = list(set(t['category'] for t in self.tickets))
            return f"No tickets found in category '{category}'. Available categories: {', '.join(available)}"
        output = f"Found {len(matching)} tickets in category '{category}':\n\n"
        for ticket in matching:
            output += f"• [{ticket['ticket_id']}] {ticket['title']} (Priority: {ticket['priority']})\n"
        return output

    def get_ticket_statistics(self, _: str = "") -> str:
        """
        Get statistics and insights about the ticket database.
        Returns:
            Statistics about tickets
        """
        total = len(self.tickets)
        # Aggregate counts for dashboard/overview style questions.
        # Using dict accumulation keeps this dependency-free and explicit.
        categories = {}
        priorities = {}
        for ticket in self.tickets:
            categories[ticket['category']] = categories.get(ticket['category'], 0) + 1
            priorities[ticket['priority']] = priorities.get(priority['priority'], 0) + 1
        output = f"Ticket Database Statistics:\n"
        output += f"Total Tickets: {total}\n\n"
        output += "By Category:\n"
        for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
            output += f"  • {cat}: {count}\n"
        output += "\nBy Priority:\n"
        for pri, count in sorted(priorities.items(), key=lambda x: x[1], reverse=True):
            output += f"  • {pri}: {count}\n"
        return output

    def get_tools(self) -> List[Tool]:
        """
        Get all tools as LangChain Tool objects.
        Returns:
            List of Tool objects for the agent
        """
        # Tool descriptions are part of the model prompt context.
        # High-quality descriptions materially improve tool-selection accuracy.
        return [
            Tool(
                name="SearchSimilarTickets",
                func=self.search_similar_tickets,
                description="""Use this tool to search for similar support tickets based on a problem description or question.
Input should be a clear description of the issue or question.
This is the PRIMARY tool for answering "how to fix" or "similar issue" questions."""
            ),
            Tool(
                name="GetTicketByID",
                func=self.get_ticket_by_id,
                description="""Use this tool to retrieve details of a specific ticket by its ID.
Input should be a ticket ID like 'TICK-001'.
Use this when the user mentions a specific ticket number."""
            ),
            Tool(
                name="SearchByCategory",
                func=self.search_by_category,
                description="""Use this tool to find all tickets in a specific category.
Input should be a category name like 'Authentication', 'Database', 'Payment', 'Mobile', etc.
Use this when the user wants to see all issues of a certain type."""
            ),
            Tool(
                name="GetTicketStatistics",
                func=self.get_ticket_statistics,
                description="""Use this tool to get statistics about the ticket database including total count,
category distribution, and priority breakdown.
No input needed - just call this tool.
Use this when the user asks for statistics, summaries, or overview of tickets."""
            )
        ]

In [35]:

# Import all depedencies for Langraph

import os

from langchain_openai import ChatOpenAI

from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage





In [36]:

# Fetch the API key from Colab secrets
openai_api_key = userdata.get('OPENAI_APIKEY')

# Set up Environmental Variables
os.environ["OPENAI_API_KEY"] = openai_api_key

In [7]:

# ============================================================================
# PART 1: Setup LangChain Agent & Tools
# ============================================================================
print("\n" + "="*80)
print("PART 1: Setting Up the LangChain Agent")
print("="*80)




PART 1: Setting Up the LangGraph Agent


In [37]:
print("\nInitializing LLM...")
llm = ChatOpenAI(
    model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
    temperature=0,
    api_key=os.getenv('OPENAI_API_KEY')
)
print("✓ LLM initialized")


Initializing LLM...
✓ LLM initialized


In [38]:
print("\nCreating agent tools...")
tool_manager = SupportTicketTools()
tools = tool_manager.get_tools()


Creating agent tools...


In [10]:
# Bind tools natively using LangChain's bind_tools (no manual JSON schema required)
llm_with_tools = llm.bind_tools(tools)

print(f"✓ Created {len(tools)} tools:")
for tool in tools:
    print(f"  • {tool.name}: {tool.description.split('.')[0]}")

✓ Created 4 tools:
  • SearchSimilarTickets: Use this tool to search for similar support tickets based on a problem description or question
  • GetTicketByID: Use this tool to retrieve details of a specific ticket by its ID
  • SearchByCategory: Use this tool to find all tickets in a specific category
  • GetTicketStatistics: Use this tool to get statistics about the ticket database including total count,
category distribution, and priority breakdown


In [39]:
print("\nCreating agent tools...")
tool_manager = SupportTicketTools()
tools = tool_manager.get_tools()


Creating agent tools...


In [40]:
# Construct System and User Prompt Template

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert support desk assistant that helps troubleshoot technical issues.

You have access to a database of previous support tickets with their resolutions.
Use your tools to find relevant information and provide helpful, accurate answers.

Guidelines:
- ALWAYS search for similar tickets when asked about troubleshooting or "how to fix" questions
- Be specific and reference ticket IDs when providing solutions
- If multiple similar issues exist, mention the most relevant ones
- Admit when you don't have enough information
- Be concise but thorough in your responses
- When appropriate, use multiple tools to gather complete information

Remember: Your primary value is retrieving and applying solutions from past tickets!"""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [41]:
# Create the Tool Calling Agent using langchain_classic
agent = create_tool_calling_agent(llm, tools, prompt)

In [42]:
# Construct the AgentExecutor runner
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # Automatically prints decision trace and tool execution
    max_iterations=5,
    handle_parsing_errors=True
)

In [43]:
def run_agent(query: str) -> str:
    """Run single-turn query through the LangChain AgentExecutor."""
    result = agent_executor.invoke({"input": query})
    return result["output"]

In [44]:
# ============================================================================
# PART 2: Simple Query - RAG Tool Selection
# ============================================================================
print("\n" + "="*80)
print("PART 2: Simple Query - Agent Selects RAG Tool")
print("="*80)

query1 = "How do I fix authentication problems after password reset?"
print(f"\nQuery: '{query1}'\n")
response1 = run_agent(query1)
print("\n" +  "-" * 80)
print("FINAL ANSWER:")
print(response1)


PART 2: Simple Query - Agent Selects RAG Tool

Query: 'How do I fix authentication problems after password reset?'



> Entering new AgentExecutor chain...

Invoking: `SearchSimilarTickets` with `authentication problems after password reset`


Found similar tickets:

--- Ticket 1 ---
Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.
Resolution: Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change.
Category: Authentication
Priority: High

--- Ticket 2 ---
Ticket ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing password reset. Error 

In [45]:
# ============================================================================
# PART 3: Specific Lookup - Different Tool
# ============================================================================
print("\n" + "="*80)
print("PART 3: Specific Ticket Lookup")
print("="*80)

query2 = "Show me details of ticket TICK-005"
print(f"\nQuery: '{query2}'\n")
response2 = run_agent(query2)
print("\n" + "-" * 80)
print("FINAL ANSWER:")
print(response2)


PART 3: Specific Ticket Lookup

Query: 'Show me details of ticket TICK-005'



> Entering new AgentExecutor chain...

Invoking: `GetTicketByID` with `TICK-005`


Ticket ID: TICK-005
Title: Memory leak in background worker process
Description: Background job processor showing steady memory increase over time. Process killed by OS after reaching 8GB memory usage. Requires daily restart to maintain functionality.
Resolution: Memory profiling revealed unclosed database cursors in batch processing loop. Added explicit cursor.close() calls and implemented context managers for all DB operations. Memory usage now stable at 500MB. Set up memory usage monitoring alerts.
Category: Performance
Priority: High
Created: 2024-01-25
Resolved: 2024-01-26Here are the details for ticket **TICK-005**:

- **Title**: Memory leak in background worker process
- **Description**: The background job processor was showing a steady increase in memory usage over time, leading to the process being killed by the OS a

In [46]:
# ============================================================================
# PART 4: Category Filtering
# ============================================================================
print("\n" + "="*80)
print("PART 4: Category-Based Search")
print("="*80)

query3 = "What payment-related issues have we seen?"
print(f"\nQuery: '{query3}'\n")
response3 = run_agent(query3)
print("\n" + "-" * 80)
print("FINAL ANSWER:")
print(response3)


PART 4: Category-Based Search

Query: 'What payment-related issues have we seen?'



> Entering new AgentExecutor chain...

Invoking: `SearchByCategory` with `Payment`


Found 1 tickets in category 'Payment':

• [TICK-003] Payment processing fails for international cards (Priority: High)
We have one payment-related issue recorded:

- **Ticket ID:** TICK-003
  - **Issue:** Payment processing fails for international cards
  - **Priority:** High

If you need more details about this ticket or its resolution, please let me know!

> Finished chain.

--------------------------------------------------------------------------------
FINAL ANSWER:
We have one payment-related issue recorded:

- **Ticket ID:** TICK-003
  - **Issue:** Payment processing fails for international cards
  - **Priority:** High

If you need more details about this ticket or its resolution, please let me know!


In [47]:
# ============================================================================
# PART 6: Multi-Step Reasoning
# ============================================================================
print("\n" + "="*80)
print("PART 6: Multi-Step Reasoning")
print("="*80)

query5 = "Find database-related critical issues and tell me how they were resolved"
print(f"\nQuery: '{query5}'\n")
response5 = run_agent(query5)
print("\n" + "-" * 80)
print("FINAL ANSWER:")
print(response5)


PART 6: Multi-Step Reasoning

Query: 'Find database-related critical issues and tell me how they were resolved'



> Entering new AgentExecutor chain...

Invoking: `SearchByCategory` with `Database`


Found 2 tickets in category 'Database':

• [TICK-002] Database connection timeout in production (Priority: Critical)
• [TICK-021] Database connection excedding in production (Priority: Critical)

Invoking: `GetTicketByID` with `TICK-002`


Ticket ID: TICK-002
Title: Database connection timeout in production
Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.
Resolution: Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connection pooling monitoring and alerts. Optimized long-running queries that were holding connections.
Category: Database
Priority: Critical
Created: 2024-01-18
Resolved: 2

In [48]:
# ============================================================================
# PART 7: Conversational Agent with Memory
# ============================================================================
print("\n" + "="*80)
print("PART 7: Conversational Agent with Memory")
print("="*80)

print("\nSimulating a multi-turn conversation...")

def run_conversational_agent(chat_history: list, query: str) -> tuple:
    """
    Run the AgentExecutor while passing the persistent `chat_history`.
    """
    response = agent_executor.invoke({
        "input": query,
        "chat_history": chat_history
    })

    # Append turns back to chat_history
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=response["output"]))

    return chat_history, response["output"]

# Initialize conversation history state
conversation_history = []

print("\n--- Conversation Turn 1 ---")
conv_query1 = "What issues have we had with iOS?"
print(f"User: {conv_query1}\n")
conversation_history, conv_response1 = run_conversational_agent(conversation_history, conv_query1)
print(f"\nAssistant: {conv_response1}")

print("\n--- Conversation Turn 2 (Follow-up) ---")
conv_query2 = "What was the ticket ID for that?"
print(f"User: {conv_query2}\n")
conversation_history, conv_response2 = run_conversational_agent(conversation_history, conv_query2)
print(f"\nAssistant: {conv_response2}")

print("\n--- Conversation Turn 3 (Another Follow-up) ---")
conv_query3 = "How was it resolved?"
print(f"User: {conv_query3}\n")
conversation_history, conv_response3 = run_conversational_agent(conversation_history, conv_query3)
print(f"\nAssistant: {conv_response3}")


PART 7: Conversational Agent with Memory

Simulating a multi-turn conversation...

--- Conversation Turn 1 ---
User: What issues have we had with iOS?



> Entering new AgentExecutor chain...

Invoking: `SearchByCategory` with `Mobile`


Found 1 tickets in category 'Mobile':

• [TICK-004] Mobile app crashes on iOS 17 (Priority: Critical)
We have had one reported issue with iOS:

- **Ticket ID:** TICK-004
  - **Issue:** Mobile app crashes on iOS 17
  - **Priority:** Critical

If you need more details about this specific ticket or its resolution, please let me know!

> Finished chain.

Assistant: We have had one reported issue with iOS:

- **Ticket ID:** TICK-004
  - **Issue:** Mobile app crashes on iOS 17
  - **Priority:** Critical

If you need more details about this specific ticket or its resolution, please let me know!

--- Conversation Turn 2 (Follow-up) ---
User: What was the ticket ID for that?



> Entering new AgentExecutor chain...
The ticket ID for the reported issue with the